# Tokens and Context 

In [1]:
import sys
sys.path.insert(0, "..")
from config import API_KEY as key, ENDPOINT_BASE as endpoint_base


In [2]:
! pip install tiktoken numpy -q

In [1]:
from dataclasses import dataclass
from typing import List, Dict, Any, Callable, Tuple
import numpy as np
import tiktoken

@dataclass
class Chunk:
    id: str
    text: str
    meta: Dict[str, Any]

def chunk_text_tokens(
    text: str,
    *,
    chunk_tokens: int = 600,
    overlap_tokens: int = 120,
    encoding_name: str = "cl100k_base",
    doc_id: str = "doc1",
) -> List[Chunk]:
    """
    Token-based chunking with overlap.
    This matches the Section 4 motivation: don't send the whole doc; send the right chunk.
    """
    enc = tiktoken.get_encoding(encoding_name)
    toks = enc.encode(text)

    if chunk_tokens <= 0:
        raise ValueError("chunk_tokens must be > 0")
    if overlap_tokens < 0 or overlap_tokens >= chunk_tokens:
        raise ValueError("overlap_tokens must be >= 0 and < chunk_tokens")

    chunks: List[Chunk] = []
    start = 0
    i = 0

    while start < len(toks):
        end = min(start + chunk_tokens, len(toks))
        chunk_toks = toks[start:end]
        chunk_text = enc.decode(chunk_toks)

        chunks.append(
            Chunk(
                id=f"{doc_id}_chunk_{i}",
                text=chunk_text,
                meta={
                    "doc_id": doc_id,
                    "token_start": start,
                    "token_end": end,
                },
            )
        )

        if end == len(toks):
            break

        start = end - overlap_tokens
        i += 1

    return chunks

def cosine_sim(a: np.ndarray, b: np.ndarray) -> float:
    denom = (np.linalg.norm(a) * np.linalg.norm(b))
    if denom == 0:
        return 0.0
    return float(np.dot(a, b) / denom)

class SimpleVectorIndex:
    """
    Tiny in-memory vector store for demo purposes.
    Replace with FAISS / pgvector / Milvus / etc in real deployments.
    """
    def __init__(self, embed_fn: Callable[[str], np.ndarray]):
        self.embed_fn = embed_fn
        self.chunks: List[Chunk] = []
        self.vectors: List[np.ndarray] = []

    def add(self, chunks: List[Chunk]) -> None:
        for ch in chunks:
            v = self.embed_fn(ch.text)
            self.chunks.append(ch)
            self.vectors.append(v.astype(np.float32))

    def search(self, query: str, top_k: int = 5) -> List[Tuple[float, Chunk]]:
        qv = self.embed_fn(query).astype(np.float32)
        scored = []
        for v, ch in zip(self.vectors, self.chunks):
            scored.append((cosine_sim(qv, v), ch))
        scored.sort(key=lambda x: x[0], reverse=True)
        return scored[:top_k]

# -----------------------------
# Embedding function
# -----------------------------
def placeholder_embed(text: str, dim: int = 384) -> np.ndarray:
    """
    Placeholder embedding function.
    Replace this with:
      - a hosted embedding model call, or
      - a local embedding model (e.g., sentence-transformers).
    """
    # deterministic "fake" embedding from hash (demo-only)
    rng = np.random.default_rng(abs(hash(text)) % (2**32))
    v = rng.normal(size=dim)
    v = v / np.linalg.norm(v)
    return v



Created 374 chunks

Top retrieved chunks:

- score=0.183 id=basic_fantasy_rules_chunk_179 tokens=700
  preview: | | Armor Class:    | 18 (m)     | 20 (m)          | 22 (m)      | | Hit Dice:       | 8*         | 12* (+10)       | 16* (+12)   | | No. of Attacks: | 1          | 1               | 1           | | Damage:         | 1d12       | 2d8             | 3d6         | | Movement:       ...

- score=0.154 id=basic_fantasy_rules_chunk_221 tokens=700
  preview: old  | Armor Class:    | 13 (11)                   | |-----------------|---------------------------| | Hit Dice:       | ½ (1d4 hit points)        | | No. of Attacks: | 1 weapon                  | | Damage:         | 1d4 or by weapon          | | Movement:       | 20' Unarmored 3...

- score=0.150 id=basic_fantasy_rules_chunk_128 tokens=700
  preview:  large, powerful horses which are both bred for their size,   strength,   and   combat   ability   and trained to tolerate the sounds and stresses of battle. They are able to attac

In [ ]:
# -----------------------------
# Example usage
# -----------------------------
if __name__ == "__main__":
    # 1) Load your Docling output markdown
    md_path = "Basic-Fantasy-RPG-Rules-r142.md"
    with open(md_path, "r", encoding="utf-8") as f:
        doc_text = f.read()

    # 2) Chunk it (this is the Section 4 "design decision")
    chunks = chunk_text_tokens(
        doc_text,
        chunk_tokens=700,
        overlap_tokens=150,
        doc_id="basic_fantasy_rules",
    )
    print(f"Created {len(chunks)} chunks")

    # 3) Build an index
    index = SimpleVectorIndex(embed_fn=placeholder_embed)
    index.add(chunks)

    # 4) Retrieve relevant chunks for a question
    question = "What happens if a Thief fails an Open Locks attempt?"
    results = index.search(question, top_k=3)

    # 5) Show what you'd pass to the LLM
    print("\nTop retrieved chunks:\n")
    for score, ch in results:
        preview = ch.text[:280].replace("\n", " ")
        print(f"- score={score:.3f} id={ch.id} tokens={ch.meta['token_end']-ch.meta['token_start']}")
        print(f"  preview: {preview}...\n")

    # 6) Prompt assembly (typical RAG)
    context_blocks = []
    for score, ch in results:
        context_blocks.append(f"[{ch.id}] {ch.text}")

    rag_prompt = f"""You are a helpful assistant. Answer using ONLY the context.

QUESTION:
{question}

CONTEXT:
{chr(10).join(context_blocks)}
"""
    print("----- RAG PROMPT (send to LLM) -----")
    print(rag_prompt[:2000])


In [3]:
# Section 4 — RAG Implementation Code Cells
# Ready to paste into Jupyter notebooks
# Matches existing tech stack: LiteLLM endpoint, requests, Docling markdown output

# ============================================================
# NOTEBOOK: 03/03_ChunkingAndEmbedding.ipynb
# ============================================================

# --- Cell 4.2a: Install dependencies ---
# Facilitator note: ChromaDB is a lightweight vector store that runs
# in-process. No external infrastructure needed.

# ! pip install chromadb sentence-transformers

# --- Cell 4.2b: Load the Docling output ---
# We're picking up where Section 3 left off.
# The markdown file is the contract between ingestion and retrieval.

"""
with open("Basic-Fantasy-RPG-Rules-r142.md", "r", encoding="utf-8") as f:
    full_text = f.read()

print(f"Document length: {len(full_text):,} characters")
print(f"First 500 characters:\\n{full_text[:500]}")
"""

# --- Cell 4.2c: Chunking — the design decision ---
# Facilitator talking point:
#   "Every parameter here is a choice that affects what the model can see.
#    There is no 'correct' chunk size — only tradeoffs."
#
# We use a simple recursive character splitter here. In production,
# you'd consider section-aware or semantic chunking.

"""
def chunk_text(text, chunk_size=500, overlap=100):
    \"\"\"
    Split text into overlapping chunks.

    Args:
        text: The full document text
        chunk_size: Target size of each chunk in characters
        overlap: Number of characters shared between adjacent chunks

    Returns:
        List of text chunks
    \"\"\"
    chunks = []
    start = 0
    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap  # Step forward, minus overlap
    return chunks


# Chunk with our default parameters
chunks = chunk_text(full_text, chunk_size=500, overlap=100)

print(f"Total chunks created: {len(chunks)}")
print(f"Average chunk length: {sum(len(c) for c in chunks) / len(chunks):.0f} characters")
print(f"\\n--- Example chunk [0] ---\\n{chunks[0]}")
print(f"\\n--- Example chunk [1] ---\\n{chunks[1]}")

# Show the overlap
print(f"\\n--- Overlap between chunk 0 and chunk 1 ---")
print(f"End of chunk 0:   ...{chunks[0][-120:]}")
print(f"Start of chunk 1: {chunks[1][:120]}...")
"""

# --- Cell 4.2d: Inspect chunking boundaries ---
# Facilitator talking point:
#   "Look at where chunks split. Did we cut in the middle of a rule?
#    A table? A sentence? This is what we mean by 'chunking is a
#    design decision.'"

"""
# Show a few chunks around a known important section
# The Thief's Open Locks rule is our test case from Section 2
for i, chunk in enumerate(chunks):
    if "open locks" in chunk.lower() or "Open Locks" in chunk:
        print(f"\\n=== Chunk {i} (contains 'Open Locks') ===")
        print(chunk)
        print(f"\\nLength: {len(chunk)} chars")
"""

# --- Cell 4.2e: Experiment — what happens with different chunk sizes? ---
# Facilitator talking point:
#   "This is not optimization. This is understanding the tradeoff."

"""
for size in [200, 500, 1000, 2000]:
    test_chunks = chunk_text(full_text, chunk_size=size, overlap=int(size * 0.2))
    print(f"Chunk size: {size:>5}  |  Overlap: {int(size*0.2):>4}  |  Total chunks: {len(test_chunks):>4}")
"""


# ============================================================
# NOTEBOOK: 03/04_VectorStoreAndRetrieval.ipynb
# ============================================================

# --- Cell 4.3a: Create embeddings and load into ChromaDB ---
# Facilitator talking point:
#   "We are now using TWO models. The embedding model finds the right chunks.
#    The language model reasons over them. These are separate jobs."
#
# NOTE: This uses sentence-transformers locally for embeddings.
# If your MaaS instance has nomic-embed-text-v1-5, you can use that
# via the API instead (alternative shown below).

"""
import chromadb
from chromadb.utils import embedding_functions

# Option A: Use a local embedding model (no API call needed)
# This downloads ~90MB on first run
embedding_fn = embedding_functions.SentenceTransformerEmbeddingFunction(
    model_name="all-MiniLM-L6-v2"
)

# Create an in-memory ChromaDB instance
client = chromadb.Client()

# Create a collection (like a table for vectors)
collection = client.create_collection(
    name="rpg_rules",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}  # Use cosine similarity
)

# Add chunks to the collection
collection.add(
    documents=chunks,
    ids=[f"chunk_{i}" for i in range(len(chunks))],
    metadatas=[{"chunk_index": i, "source": "Basic-Fantasy-RPG-Rules-r142.pdf"} for i in range(len(chunks))]
)

print(f"Loaded {collection.count()} chunks into the vector store")
"""

# --- Cell 4.3a-alt: Alternative — use MaaS embedding endpoint ---
# Uncomment this if nomic-embed-text-v1-5 is available on your MaaS instance

"""
import requests
import chromadb
import numpy as np

# Custom embedding function that calls MaaS
class MaaSEmbeddingFunction:
    def __init__(self, api_key, base_url, model="nomic-embed-text-v1-5"):
        self.api_key = api_key
        self.base_url = base_url
        self.model = model

    def __call__(self, input):
        url = f"{self.base_url}/embeddings"
        headers = {
            "Authorization": f"Bearer {self.api_key}",
            "Content-Type": "application/json"
        }
        data = {
            "model": self.model,
            "input": input
        }
        response = requests.post(url, headers=headers, json=data)
        response.raise_for_status()
        result = response.json()
        return [item["embedding"] for item in result["data"]]

embedding_fn = MaaSEmbeddingFunction(api_key=key, base_url=endpoint_base)

client = chromadb.Client()
collection = client.create_collection(
    name="rpg_rules",
    embedding_function=embedding_fn,
    metadata={"hnsw:space": "cosine"}
)

# Add in batches to avoid overwhelming the API
batch_size = 50
for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i+batch_size]
    collection.add(
        documents=batch,
        ids=[f"chunk_{j}" for j in range(i, i+len(batch))],
        metadatas=[{"chunk_index": j, "source": "Basic-Fantasy-RPG-Rules-r142.pdf"} for j in range(i, i+len(batch))]
    )
    print(f"  Added chunks {i} to {i+len(batch)-1}")

print(f"\\nLoaded {collection.count()} chunks into the vector store")
"""


# --- Cell 4.3b: Test retrieval — the moment of truth ---
# Facilitator talking point:
#   "We haven't changed the model. We haven't tuned anything.
#    We're just asking: can we find the right paragraph?"

"""
question = "What happens if a Thief fails an Open Locks attempt?"

# Retrieve the top 3 most relevant chunks
results = collection.query(
    query_texts=[question],
    n_results=3
)

print(f"Question: {question}\\n")
for i, (doc, distance) in enumerate(zip(results['documents'][0], results['distances'][0])):
    similarity = 1 - distance  # ChromaDB returns distance; lower = more similar for cosine
    print(f"--- Retrieved Chunk {i+1} (similarity: {similarity:.3f}) ---")
    print(doc)
    print()
"""


# --- Cell 4.3c: Inspect what was retrieved ---
# Facilitator talking point:
#   "Before we send anything to the model, look at what came back.
#    Is the answer in here? If retrieval fails, no model can save you."

"""
# Check: does the retrieved context contain the actual answer?
expected_phrase = "must wait until they have gained another level"
found_in_context = any(expected_phrase.lower() in doc.lower() for doc in results['documents'][0])
print(f"Expected answer phrase found in retrieved context: {found_in_context}")

if found_in_context:
    print("✅ Retrieval found the relevant chunk. The model has what it needs.")
else:
    print("❌ Retrieval missed the relevant chunk. The model will likely fail.")
    print("   This is a retrieval problem, not a model problem.")
"""


# ============================================================
# NOTEBOOK: 03/05_RAGQuery.ipynb
# ============================================================

# --- Cell 4.4a: The RAG query function ---
# This is where ingestion + retrieval + generation come together.
# Facilitator talking point:
#   "Notice what we're doing: retrieve first, then generate.
#    The model never sees the whole document. Just the relevant pieces."

"""
import requests
import json

def rag_query(question, collection, api_key, base_url,
              model="granite-3-2-8b-instruct", n_results=3, max_tokens=300):
    \"\"\"
    Full RAG pipeline: retrieve context, then generate a grounded answer.

    Args:
        question: The user's question
        collection: ChromaDB collection with document chunks
        api_key: MaaS API key
        base_url: MaaS API base URL
        model: Model to use for generation
        n_results: Number of chunks to retrieve
        max_tokens: Max tokens in the generated answer

    Returns:
        dict with 'answer', 'retrieved_chunks', and 'model' keys
    \"\"\"
    # Step 1: Retrieve relevant chunks
    results = collection.query(
        query_texts=[question],
        n_results=n_results
    )
    retrieved_chunks = results['documents'][0]

    # Step 2: Build the grounded prompt
    context = "\\n\\n---\\n\\n".join(retrieved_chunks)

    messages = [
        {
            "role": "system",
            "content": (
                "You are a helpful assistant. Answer the question using ONLY "
                "the provided context. If the context does not contain enough "
                "information to answer, say so explicitly. Do not make up information."
            )
        },
        {
            "role": "user",
            "content": f"Context:\\n{context}\\n\\nQuestion: {question}"
        }
    ]

    # Step 3: Generate the answer
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    data = {
        "model": model,
        "messages": messages,
        "max_tokens": max_tokens,
        "temperature": 0  # Deterministic for factual Q&A
    }

    url_chat = f"{base_url}/chat/completions"
    response = requests.post(url_chat, headers=headers, json=data)
    response.raise_for_status()

    answer = response.json()["choices"][0]["message"]["content"]

    return {
        "question": question,
        "answer": answer,
        "retrieved_chunks": retrieved_chunks,
        "model": model
    }
"""


# --- Cell 4.4b: Test RAG with our known question ---
# Facilitator talking point:
#   "Same model. Same question. Different result.
#    What changed? Not the model — the context."

"""
result = rag_query(
    question="What happens if a Thief fails an Open Locks attempt?",
    collection=collection,
    api_key=key,
    base_url=endpoint_base
)

print(f"Question: {result['question']}\\n")
print(f"RAG Answer:\\n{result['answer']}\\n")
print(f"Model: {result['model']}")
print(f"Chunks used: {len(result['retrieved_chunks'])}")
"""

# Expected output (or close to it):
# "If a Thief fails an Open Locks attempt, they must wait until they
#  have gained another level of experience before trying again."


# --- Cell 4.4c: Side-by-side comparison — baseline vs. RAG ---
# THIS IS THE KEY TEACHING MOMENT.
# Facilitator talking point:
#   "This is why we established a baseline in Section 2.
#    Without 'before,' you can't prove 'after.'"

"""
questions = [
    "What happens if a Thief fails an Open Locks attempt?",
    "Why can't Elves roll higher than a d6 for hit points?",
    "What is the maximum number of retainers a character can have?",
    "How does turning undead work for Clerics?",
]

print("=" * 80)
print("BASELINE vs RAG COMPARISON")
print("=" * 80)

url_chat = f"{endpoint_base}/chat/completions"
headers = {
    "Authorization": f"Bearer {key}",
    "Content-Type": "application/json"
}

for q in questions:
    print(f"\\n{'─' * 80}")
    print(f"QUESTION: {q}")
    print(f"{'─' * 80}")

    # Baseline: no context
    baseline_data = {
        "model": "granite-3-2-8b-instruct",
        "messages": [{"role": "user", "content": q}],
        "max_tokens": 200,
        "temperature": 0
    }
    try:
        baseline_resp = requests.post(url_chat, headers=headers, json=baseline_data)
        baseline_answer = baseline_resp.json()["choices"][0]["message"]["content"]
    except Exception as e:
        baseline_answer = f"[Error: {e}]"

    # RAG: with retrieved context
    try:
        rag_result = rag_query(q, collection, key, endpoint_base)
        rag_answer = rag_result["answer"]
    except Exception as e:
        rag_answer = f"[Error: {e}]"

    print(f"\\n📭 BASELINE (no context):\\n{baseline_answer}")
    print(f"\\n📬 RAG (with retrieval):\\n{rag_answer}")

print(f"\\n{'=' * 80}")
print("Review: Which answers improved? Which didn't? Why?")
print("=" * 80)
"""


# --- Cell 4.4d: Make retrieval visible ---
# Don't just show the answer — show what was retrieved.
# This builds the "defensible" quality the customer needs (Section 1.4).
# Facilitator talking point:
#   "This is how you answer 'why did the system say that?'
#    You can point to the exact chunks."

"""
question = "What happens if a Thief fails an Open Locks attempt?"
result = rag_query(question, collection, key, endpoint_base)

print(f"Question: {result['question']}\\n")
print(f"Answer: {result['answer']}\\n")
print("=" * 60)
print("EVIDENCE — Retrieved chunks that informed this answer:")
print("=" * 60)
for i, chunk in enumerate(result['retrieved_chunks']):
    print(f"\\n--- Chunk {i+1} ---")
    print(chunk)
"""


# ============================================================
# OPTIONAL: Cell 4.5 — Chunking comparison experiment
# ============================================================
# Facilitator note: Use this if time permits. It demonstrates that
# chunking strategy directly impacts answer quality.

"""
# Compare RAG results with different chunk sizes
question = "What happens if a Thief fails an Open Locks attempt?"

for chunk_size in [200, 500, 1000]:
    print(f"\\n{'=' * 60}")
    print(f"CHUNK SIZE: {chunk_size} characters")
    print(f"{'=' * 60}")

    # Re-chunk
    test_chunks = chunk_text(full_text, chunk_size=chunk_size, overlap=int(chunk_size * 0.2))

    # Create a fresh collection
    test_collection = client.get_or_create_collection(
        name=f"rpg_rules_{chunk_size}",
        embedding_function=embedding_fn,
        metadata={"hnsw:space": "cosine"}
    )

    # Populate
    test_collection.add(
        documents=test_chunks,
        ids=[f"chunk_{i}" for i in range(len(test_chunks))]
    )

    # Query
    result = rag_query(question, test_collection, key, endpoint_base)
    print(f"Chunks in store: {len(test_chunks)}")
    print(f"Answer: {result['answer']}")

    # Cleanup
    client.delete_collection(f"rpg_rules_{chunk_size}")
"""


# ============================================================
# NOTES FOR FACILITATOR: Pre-built outputs
# ============================================================
#
# If live execution stalls, here are the expected outputs:
#
# Chunk count (500 char, 100 overlap): ~650 chunks (depends on doc length)
#
# Retrieval for "Open Locks" question: Should return chunks containing
# the Thief class abilities section, specifically the paragraph:
#   "Open Locks allows the Thief to unlock a lock without a proper key.
#    It may only be tried once per lock. If the attempt fails, the Thief
#    must wait until they have gained another level of experience before
#    trying again."
#
# RAG answer should closely match the source text above.
#
# Baseline answer will reference D&D or generic RPG rules.
#
# The side-by-side comparison should show clear improvement for
# questions with explicit answers in the rulebook, and may show
# less improvement for questions requiring implicit reasoning.

'\n# Compare RAG results with different chunk sizes\nquestion = "What happens if a Thief fails an Open Locks attempt?"\n\nfor chunk_size in [200, 500, 1000]:\n    print(f"\\n{\'=\' * 60}")\n    print(f"CHUNK SIZE: {chunk_size} characters")\n    print(f"{\'=\' * 60}")\n\n    # Re-chunk\n    test_chunks = chunk_text(full_text, chunk_size=chunk_size, overlap=int(chunk_size * 0.2))\n\n    # Create a fresh collection\n    test_collection = client.get_or_create_collection(\n        name=f"rpg_rules_{chunk_size}",\n        embedding_function=embedding_fn,\n        metadata={"hnsw:space": "cosine"}\n    )\n\n    # Populate\n    test_collection.add(\n        documents=test_chunks,\n        ids=[f"chunk_{i}" for i in range(len(test_chunks))]\n    )\n\n    # Query\n    result = rag_query(question, test_collection, key, endpoint_base)\n    print(f"Chunks in store: {len(test_chunks)}")\n    print(f"Answer: {result[\'answer\']}")\n\n    # Cleanup\n    client.delete_collection(f"rpg_rules_{chunk_size